<a href="https://colab.research.google.com/github/RegistryHJ/pico-with-data-pi/blob/main/src/examples/ex24_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 데이터 분석

### 라이브러리 임포트

In [1]:
import pandas as pd
from scipy import stats

### 데이터 불러오기

In [2]:
# CSV 파일 경로 (main.py에서 추출된 data.csv를 가상 드라이브에 업로드)
csv_file = "./data.csv"

# CSV 파일 읽어서 데이터프레임으로 변환
df = pd.read_csv(csv_file, names=["DateTime", "Temp", "Humid", "AQI", "CO2", "TVOC"])

# 데이터프레임의 선두 5개의 데이터 확인하기
df.head()

,DateTime,Temp,Humid,AQI,CO2,TVOC
0,2024-10-9 2:50:25,26.39,80.06,1,400,15
1,2024-10-9 3:5:25,29.33,53.45,1,453,48
2,2024-10-9 3:20:26,29.54,52.20,2,565,110
3,2024-10-9 3:35:26,29.31,51.99,1,432,38
4,2024-10-9 3:50:26,29.01,52.25,1,423,34


### 1. 평균(mean), 표준편차(std) 산출

In [3]:
# 데이터프레임의 평균(mean()), 표준편차(std()) 함수 사용하여 값 구하기
mean = df[["Temp", "Humid", "AQI", "CO2", "TVOC"]].mean()
std = df[["Temp", "Humid", "AQI", "CO2", "TVOC"]].std()

# 결과 확인
print(f"평균:\n{mean}\n")
print(f"표준편차:\n{std}")

평균:
Temp      29.190942
Humid     47.647522
AQI        2.663176
CO2      766.071553
TVOC     506.394415
dtype: float64

표준편차:
Temp       0.711669
Humid      4.444145
AQI        1.017326
CO2      251.458452
TVOC     781.125527
dtype: float64


### 2. 중복 데이터 행의 개수 확인

In [4]:
# 데이터 기본 통계 정보 확인하기
print(f"데이터의 기본 통계 정보:\n{df.describe()}\n")

# 중복된 데이터 행의 개수를 DateTime을 기준으로 구하기
duplicate_rows = df.duplicated(subset=["DateTime"]).sum()
print(f"중복된 데이터 행의 개수: {duplicate_rows}")

데이터의 기본 통계 정보:
             Temp       Humid         AQI          CO2         TVOC
count  573.000000  573.000000  573.000000   573.000000   573.000000
mean    29.190942   47.647522    2.663176   766.071553   506.394415
std      0.711669    4.444145    1.017326   251.458452   781.125527
min     26.390000   36.210000    1.000000   400.000000     0.000000
25%     28.690000   44.990000    2.000000   587.000000   123.000000
50%     29.060000   47.460000    3.000000   799.000000   295.000000
75%     29.730000   51.210000    3.000000   889.000000   577.000000
max     31.070000   80.060000    5.000000  2154.000000  6313.000000

중복된 데이터 행의 개수: 0


### 3. 결측값(Missing Value) 개수 확인

In [5]:
# AQI, CO2, TVOC가 결측값(0)인 경우 합산
aqi_zero_count = (df["AQI"].eq(0)).sum()
co2_zero_count = (df["CO2"].eq(0)).sum()
tvoc_zero_count = (df["TVOC"].eq(0)).sum()

# 결과 확인
print(f"AQI 값이 0인 데이터 개수: {aqi_zero_count}")
print(f"CO2 값이 0인 데이터 개수: {co2_zero_count}")
print(f"TVOC 값이 0인 데이터 개수: {tvoc_zero_count}")

AQI 값이 0인 데이터 개수: 0
CO2 값이 0인 데이터 개수: 0
TVOC 값이 0인 데이터 개수: 22


### 4. 이상값(Outlier) 확인 - Z Score 사용
Z Score(표준 점수): 값이 평균에서 3 표준편차 초과하여 벗어나면 이상치로 간주.

In [6]:
# DateTime을 제외한 나머지 값들에 대해 Z Score 구하기
z_scores = stats.zscore(df[["Temp", "Humid", "AQI", "CO2", "TVOC"]])

# 이상값은 |Z Score| > 3; 즉, 정상값은 |Z Score| <= 3
outliers = (abs(z_scores) > 3)

# 결과 확인
outliers = df[outliers.any(axis=1)]
print(f"Z Score 기준으로 이상치가 있는 데이터:\n{outliers}")

Z Score 기준으로 이상치가 있는 데이터:
                DateTime   Temp  Humid  AQI   CO2  TVOC
0      2024-10-9 2:50:25  26.39  80.06    1   400    15
88    2024-10-10 1:37:41  26.98  65.83    1   400     0
214    2024-10-11 9:8:18  28.10  52.20    5  1648  4659
289   2024-10-12 3:53:41  29.40  52.12    5  1658  4731
290    2024-10-12 4:8:41  29.46  52.67    5  1452  3351
308   2024-10-12 8:38:47  29.26  52.63    5  2131  6250
309   2024-10-12 8:53:47  29.30  52.50    5  2109  6187
310    2024-10-12 9:8:47  29.28  52.32    5  1695  4998
311   2024-10-12 9:23:47  29.29  52.24    5  1514  3750
312   2024-10-12 9:38:48  29.33  51.97    5  1638  4589
313   2024-10-12 9:53:48  29.30  51.57    5  1399  3024
319  2024-10-12 11:23:50  29.31  50.26    5  1437  3256
389   2024-10-13 4:54:11  29.18  46.32    5  2154  6313
390    2024-10-13 5:9:11  29.13  46.60    5  1753  5159
